# Lesson 01 - Introduction to AI Agents

Welcome to the first lesson in the **AI Agents for Beginners** course!

An **AI agent** is a program that uses a large language model (LLM) as its reasoning engine and can take *actions* in the real world — calling APIs, querying databases, or running code — to accomplish a goal on behalf of a user.

In this notebook you will build your first agent: a **Travel Agent** that recommends vacation destinations. Along the way you will learn how to:

1. Connect to a configured model provider using the **Microsoft Agent Framework**.
2. Give the agent a **tool** — a plain Python function it can call.
3. Run the agent and inspect its response.
4. Stream the agent's response token-by-token.

## Setup

Before running this notebook, make sure you have:

1. A configured AI provider. Use OpenAI, GitHub Models, MiniMax, an OpenAI-compatible endpoint, or Azure AI Foundry.
2. Provider credentials in `.env` as described in `00-course-setup/README.md`.
3. If using Azure, run `az login` and set:
   - `AZURE_AI_PROJECT_ENDPOINT` — your Azure AI Foundry project endpoint.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — the name of your deployed model.

The cell below installs the Python packages you need.

In [1]:
%pip install agent-framework


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import logging
import os
import asyncio
from typing import Annotated

from agent_framework import tool
import sys
from pathlib import Path

repo_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "shared").exists()), Path.cwd())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from shared.agent_provider import create_provider, describe_provider

provider = create_provider()
print(f"Provider configured: {describe_provider(provider)}")

Provider configured: OpenAI-compatible | model=qwen3.6-plus | endpoint=https://dashscope-intl.aliyuncs.com/compatible-mode/v1


## Creating Your First Agent

An agent needs two things:

- **Instructions** that tell it *who it is* and *how to behave* (a system prompt).
- **Tools** — Python functions decorated with `@tool` that the agent can call to retrieve information or perform actions.

Below we define a simple tool that returns a list of popular vacation destinations. The agent will use this tool when a user asks for travel recommendations.

In [6]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [8]:
agent = await provider.create_agent(
    tools=[get_destinations],
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

Based on the available destinations, here are my top recommendations for a warm beach vacation:

🏝️ **Bali** - A tropical paradise with stunning beaches, warm weather year-round, and a perfect mix of relaxation and adventure.

🇧🇷 **Rio de Janeiro** - Famous for iconic beaches like Copacabana and Ipanema, with a vibrant beach culture and warm climate.

🇦🇺 **Sydney** - Home to world-famous Bondi Beach and many other beautiful coastal spots, with great weather for beach activities.

🇪🇸 **Barcelona** - Offers lovely Mediterranean beaches like Barceloneta, combined with amazing food, culture, and warm weather.

🇿🇦 **Cape Town** - Features beautiful beaches with dramatic scenery, warm climate, and plenty of outdoor activities.

Do any of these catch your interest? I can help you narrow down the perfect choice based on your budget, travel dates, or specific activities you're interested in!


## Streaming Responses

For a more interactive experience you can **stream** the agent's response. Instead of waiting for the full reply, the agent yields text chunks as they are generated. This is especially useful in chat interfaces where you want to display output in real time.

In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

Tokyo is one of the world's most captivating travel destinations, offering a fascinating blend of hyper-modern innovation and centuries-old tradition. As your travel agent, I highly recommend it for travelers who want world-class food, incredibly efficient transit, safe and clean streets, and a constantly shifting landscape of neighborhoods, each with its own distinct personality.

Here’s a breakdown of what makes Tokyo special and what you can expect:

### 🏙️ Neighborhoods & Must-See Areas
* **Shibuya & Shinjuku:** The beating heart of modern Tokyo. Famous for the iconic Shibuya Crossing, towering skyscrapers

IOStream.flush timed out


, vibrant nightlife, and endless shopping. Shinjuku is also home to the stunning Shinjuku Gyoen National Garden.
* **Asakusa:** Tokyo's most traditional district. Centered around **Senso-ji Temple**, Japan's oldest Buddhist temple, with Nakamise-dori street offering classic snacks, crafts, and rickshaw rides.
* **Akihabara:** The global capital of anime, manga, and electronics. Packed with multi-story arcades, themed cafes, and vintage tech shops.
* **Harajuku & Omotesando:** A fashion and youth culture hub known for Takeshita Street's quirky boutiques, right next to Omotesando's elegant, tree-lined architecture.
* **Roppongi & Ginza:** Roppongi is known for upscale dining, nightlife, and art museums (Mori Art Museum, teamLab Planets). Ginza offers luxury shopping, historic department stores, and refined afternoon tea culture.

### 🍣 Food & Drink Scene
Tokyo holds more Michelin stars than any other city, but you don't need a luxury budget to eat incredibly well:
* **Street & Casual Foo

## Summary

In this lesson you learned how to:

- **Create a provider** that connects to the configured AI service via `create_provider()`.
- **Define a tool** using the `@tool` decorator so the agent can call your Python functions.
- **Run the agent** with a user message and print its response.
- **Stream responses** for real-time output.

In the next lesson we will explore agentic frameworks in more depth and learn how to give agents more powerful tools and multi-step reasoning capabilities.